# make_it_flash: one-layer GDN pilot

Self-contained copy of the tracked pipeline. The tokenized calibration mix runs first.
The notebook measures GPU memory first and only loads the 32B teacher if at least 66 GiB is free.


In [ ]:
import base64, os, subprocess, sys, zipfile
import importlib.metadata as package_metadata
from io import BytesIO
from pathlib import Path

BUNDLE = '...'
PROJECT = Path('/kaggle/working/make_it_flash')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BytesIO(base64.b64decode(BUNDLE))) as archive:
    archive.extractall(PROJECT)
source_path = str(PROJECT / 'src')
sys.path.insert(0, source_path)
os.environ['PYTHONPATH'] = source_path + os.pathsep + os.environ.get('PYTHONPATH', '')
versions = {name: package_metadata.version(name) for name in ('torch', 'transformers', 'datasets', 'huggingface-hub', 'accelerate', 'safetensors')}
print('Dependency versions:', versions)
ARTIFACTS = Path('/kaggle/working/artifacts')
DATA = ARTIFACTS / 'data'
CACHE = ARTIFACTS / 'cache'
GDN = ARTIFACTS / 'gdn'
import torch
GPU_FREE_GIB = sum(torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())) / (1024**3) if torch.cuda.is_available() else 0.0
GPU_READY = GPU_FREE_GIB >= 66.0
print(f'Free CUDA VRAM: {GPU_FREE_GIB:.1f} GiB; teacher pass available: {GPU_READY}')


In [ ]:
# Stage 1: small, streaming public-data sample; no 32B weights are loaded here.
subprocess.run([sys.executable, '-m', 'make_it_flash.cli', 'prepare', '--output-dir', str(DATA), '--max-tokens', '100000', '--max-seq-len', '2048'], cwd=PROJECT, check=True)


In [ ]:
# Stage 2: run automatically only if the Kaggle GPU clears the 66 GiB guard.
if GPU_READY:
    subprocess.run([sys.executable, '-m', 'make_it_flash.cli', 'cache', '--data-file', str(DATA / 'calibration.jsonl'), '--output-dir', str(CACHE), '--layers', '0'], cwd=PROJECT, check=True)
else:
    print('Teacher cache skipped before weight download; use the documented HF A100 fallback.')


In [ ]:
# Stage 3: fit only when the teacher cache was created.
if (CACHE / 'cache_manifest.json').is_file():
    subprocess.run([sys.executable, '-m', 'make_it_flash.cli', 'fit', '--cache-dir', str(CACHE), '--output-dir', str(GDN), '--layer', '0'], cwd=PROJECT, check=True)
else:
    print('GDN fitting skipped; no teacher cache exists.')
